In [ ]:
# Tools
def fake_search(query):
    print(f"  [fake_search] searching: {query}")
    return ["page_A", "page_B", "page_C"]

def fake_read(url):
    print(f"  [fake_read] reading: {url}")
    return f"fake content from {url}, 500 chars"

In [ ]:
# llm
def decide(state):
    searched = any(s["action"] == "SEARCH" for s in state)
    reads_done = sum(1 for s in state if s["action"] == "READ")

    if not searched:
        return {"action": "SEARCH", "query": "gold price today"}
    elif reads_done < 2:
        return {"action": "READ", "url": f"page_{reads_done + 1}"}
    else:
        return {"action": "FINISH", "report": f"Report based on {reads_done} pages."}

In [3]:
# Agent
state = []
decision = decide(state) # llm
print(decision)

{'action': 'SEARCH', 'query': 'gold price today'}


In [4]:
# Agent
result = fake_search(decision["query"])
state.append({"action": "SEARCH", "query": decision["query"], "result": result})
state

  [fake_search] searching: gold price today


[{'action': 'SEARCH',
  'query': 'gold price today',
  'result': ['page_A', 'page_B', 'page_C']}]

In [7]:
# Agent
decision = decide(state) # llm
print(decision)

{'action': 'READ', 'url': 'page_1'}


In [8]:
# Agent
MAX_STEPS = 5
state = []
report = None

for step in range(1, MAX_STEPS + 1):
    decision = decide(state)
    print(f"STEP {step}: {decision['action']}")

    if decision["action"] == "SEARCH":
        result = fake_search(decision["query"])
        state.append({"action": "SEARCH", "query": decision["query"], "result": result})
    elif decision["action"] == "READ":
        result = fake_read(decision["url"])
        state.append({"action": "READ", "url": decision["url"], "result": result})
    elif decision["action"] == "FINISH":
        report = decision["report"]
        break

if report is None:
    report = "Step limit reached before finishing."

print("\n--- FINAL REPORT ---")
print(report)

STEP 1: SEARCH
  [fake_search] searching: gold price today
STEP 2: READ
  [fake_read] reading: page_1
STEP 3: READ
  [fake_read] reading: page_2
STEP 4: FINISH

--- FINAL REPORT ---
Report based on 2 pages.


In [9]:
state

[{'action': 'SEARCH',
  'query': 'gold price today',
  'result': ['page_A', 'page_B', 'page_C']},
 {'action': 'READ',
  'url': 'page_1',
  'result': 'fake content from page_1, 500 chars'},
 {'action': 'READ',
  'url': 'page_2',
  'result': 'fake content from page_2, 500 chars'}]